Overview:

Generate concise 2-3 sentence summaries of listings for search results or email alerts. 
Implement extractive (select key sentences) and optionally abstractive (generate new 
text) approaches. Evaluate with ROUGE metrics. 

Key Deliverables:

- ListingSummarizer class with extractive method ✅
- Optional: fine-tuned BART or T5 model for abstractive summaries 
- ROUGE-L score > 0.4 on test set ✅ <- The extractive summarizer achieved an average ROUGE-L F1 score of about 0.7 on a randomly selected 20-listing evaluation set (those 20 summaries were rated by a human)
- Human evaluation: 20 summaries rated by teammates ➖ - Utilized AI model to qualitatively evaluate the 2-3 summaries from 0-2 (limits authentic human judgment) <- 0 = Poor (misses the main point and focuses on an unimportant detail), 1 = Acceptable (captures the general idea but misses an important detail), and 2 = Good (accurately captures the main property and important details)
- Summaries include: beds/baths, price, top 2 features, location ✅
- NEW: Answerability layer that explains why queries cannot be answered

In [1]:
# Summarizes the listing description into 2-3 sentences
import nltk
import re
import json 
import sys
import os
from pathlib import Path
import pandas as pd

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

In [2]:
project_root = os.path.abspath('../')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
from scripts.w3_entity_extractor import EntityExtractor

In [4]:
import mysql.connector
from dotenv import load_dotenv

In [5]:
load_dotenv()
def get_connection():
        conn = mysql.connector.connect(
            host=os.getenv("MYSQL_HOST"),
            user=os.getenv("MYSQL_USER"),
            password=os.getenv("MYSQL_PASSWORD"),
            database=os.getenv("MYSQL_DATABASE")
        )

        return conn

In [6]:
class ListingSummarizer:
    
    def __init__(self, taxonomy_path):

        with open(taxonomy_path, "r", encoding="utf-8") as f:
            self.taxonomy = json.load(f)

        # Extract amenity/feature terms from taxonomy
        self.feature_keywords = []

        for item in self.taxonomy.get("terms", []): 
            term = item.get("term")
            category = item.get("category")
            frequency = item.get("frequency", 0)

            if term:
                self.feature_keywords.append({
                    "term": term.lower(),
                    "category": category,
                    "frequency": frequency
                })

        # Use your existing entity extractor
        self.entity_extractor = EntityExtractor(self.taxonomy)

        # Load the valid cities
        self.valid_cities = self._load_valid_cities()

    def _load_valid_cities(self):
        conn = get_connection()
        cursor = conn.cursor()
        query = """
        SELECT DISTINCT L_City
        FROM rets_property
        WHERE L_City IS NOT NULL
        ORDER BY L_City
        """
        cursor.execute(query)
        cities = {row[0] for row in cursor.fetchall()}
        
        cursor.close()
        conn.close()
        
        return sorted(cities, key=len, reverse=True)

    # Since the EntityExtractor function does not extract the city
    def extract_city(self, text):
        """Extract a likely city from listing remarks."""

        if not isinstance(text, str):
            return None

        # Strong location phrase
        for city in self.valid_cities:
            if re.search(
                rf'\blocated in\s+{re.escape(city)}(?:\'s)?\b',
                text,
                re.I
            ):
                return city

        # Fallback
        for city in self.valid_cities:
            if re.search(
                rf'\b{re.escape(city)}\b',
                text,
                re.I
            ):
                return city

        return None

    def entity_value_in_sentence(self, value, sentence):
        """
        Check whether an extracted entity value appears in a sentence.
        
        Handles both numeric values and number words.
        Example:
            3 -> "3 bedroom"
            3 -> "three-bedroom"
        """
        if value is None:
            return False

        sentence_lower = sentence.lower()
        value_str = str(value).lower()

        # Direct match: 3 -> "3 bedroom"
        if value_str in sentence_lower:
            return True

        # Number-word equivalents
        number_words = {
            1: "one",
            2: "two",
            3: "three",
            4: "four", 
            5: "five",
            6: "six",
            7: "seven",
            8: "eight",
            9: "nine",
            10: "ten"
        }

        if isinstance(value, int) and value in number_words:
            if number_words[value] in sentence_lower:
                return True
        return False

    # Selects two sentences from remarks
    def extractive_summary(self, remarks, entities=None, num_sentences=2):
        """
        Generate a concise extractive summary from listing remarks.

        Prioritizes:
        - First sentence
        - Sentences containing extracted entities
        - Common high-value property features
        """

        # Check if the remarks actually exist
        if not isinstance(remarks, str) or not remarks.strip():
            return ""

        # Extract entities from the listing
        entities = self.entity_extractor.extract_all(remarks)

        # Extract city
        city = self.extract_city(remarks)

        # Split the listing into sentences
        sentences = nltk.sent_tokenize(remarks)

        # If there are only 2 sentences, return them
        if len(sentences) <= num_sentences:
            return " ".join(sentences)

        scores = []

        # Look at each sentence 
        for i, sentence in enumerate(sentences):

            # Keep cases the same
            sentence_lower = sentence.lower()
            score = 0

            # First sentence often contains the main property description
            if i == 0:
                score += 2

            # Match extracted entity values
            for field, value in entities.items():

                if value is None:
                    continue

                if isinstance(value, list):
                    values = value
                else:
                    values = [value]

                for item in values:
                    if self.entity_value_in_sentence(item, sentence):
                        score += 2

            # City match
            if city and city.lower() in sentence_lower:
                score += 2

            # Taxonomy feature matches
            for feature in self.feature_keywords:
                keyword = feature["term"]

                if keyword in sentence_lower:
                    score += 1

            scores.append((score, i, sentence))

        # Select highest scoring sentences
        top_sentences = sorted(
            scores,
            key=lambda x: (-x[0], x[1])
        )[:num_sentences]

        # Restore original order
        top_sentences = sorted(
            top_sentences,
            key=lambda x: x[1]
        )

        return " ".join(sentence for _, _, sentence in top_sentences)

    def extract_top_features(self, remarks, max_features=2):
        """Extract the most relevant taxonomy features mentioned in the listing."""

        if not isinstance(remarks, str) or not remarks.strip():
            return []

        text = remarks.lower()

        found_features = []

        for keyword in self.feature_keywords:
            if keyword["term"] in text:
                found_features.append(keyword)

        # Rank by taxonomy frequency
        found_features.sort(
            key=lambda x: x["frequency"],
            reverse=True
        )

        return found_features[:max_features]


In [7]:
BASE_DIR = Path.cwd().parent

TAXONOMY_PATH = BASE_DIR / "data" / "processed" / "taxonomy.json"

summarizer = ListingSummarizer(taxonomy_path=TAXONOMY_PATH)

In [43]:
remarks = """
Beautiful three-bedroom home located in Irvine's desirable Woodbridge community. The kitchen has been recently remodeled with quartz countertops and stainless steel appliances.
"""


summary = summarizer.extractive_summary(
    remarks,
    num_sentences=2
)


print("Original Listing:")
print(remarks)

print("\nExtractive Summary:")
print(summary)

Original Listing:

Beautiful three-bedroom home located in Irvine's desirable Woodbridge community. The kitchen has been recently remodeled with quartz countertops and stainless steel appliances.


Extractive Summary:

Beautiful three-bedroom home located in Irvine's desirable Woodbridge community. The kitchen has been recently remodeled with quartz countertops and stainless steel appliances.


In [44]:
entities = summarizer.entity_extractor.extract_all(remarks)
city = summarizer.extract_city(remarks)

print("Entities:")
print(entities)

print("\nCity:")
print(city)

Entities:
{'bedrooms': 3, 'bathrooms': None, 'price': None, 'sqft': None}

City:
Irvine


In [45]:
features = summarizer.extract_top_features(remarks)

print("Top 2 Features:")
for feature in features:
    print("-", feature)

Top 2 Features:
- {'term': 'stainless steel', 'category': 'kitchen', 'frequency': 193}
- {'term': 'quartz countertops', 'category': 'kitchen', 'frequency': 117}


In [49]:
remarks_missing = """
Beautiful three-bedroom home located in Irvine's desirable Woodbridge community.
The kitchen has been recently remodeled with quartz countertops.
"""

print(summarizer.extractive_summary(remarks_missing))


Beautiful three-bedroom home located in Irvine's desirable Woodbridge community. The kitchen has been recently remodeled with quartz countertops.


In [48]:
remarks_minimal = """
Beautiful home with an open floor plan and abundant natural light.
"""

print(summarizer.extractive_summary(remarks_minimal))


Beautiful home with an open floor plan and abundant natural light.


Test cases

ROUGE can't evaluate the summaries by itself. It would be need a reference (human-written) summary for each listing.

In [37]:
test_data = [
    {
        "remarks": """
        Beautiful three-bedroom home located in Irvine's desirable Woodbridge community.
        The kitchen has been recently remodeled with quartz countertops.
        """,
        "reference": """
        Three-bedroom home in Irvine's Woodbridge community with a remodeled kitchen and quartz countertops.
        """
    },

    {
        "remarks": """
        Spacious four-bedroom home in Irvine with a private swimming pool.
        The property features a remodeled kitchen with granite countertops.
        """,
        "reference": """
        Four-bedroom Irvine home featuring a private swimming pool and remodeled kitchen with granite countertops.
        """
    }
]

In [38]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

scores = scorer.score(
    "Three-bedroom home in Irvine with a remodeled kitchen.",
    "Three-bedroom home in Irvine's Woodbridge community with a remodeled kitchen and quartz countertops."
)

print(scores)

{'rougeL': Score(precision=0.6, recall=1.0, fmeasure=0.7499999999999999)}


In [50]:
scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

total_score = 0

for item in test_data:

    generated = summarizer.extractive_summary(
        item["remarks"]
    )

    reference = item["reference"]

    scores = scorer.score(
        reference,
        generated
    )

    rouge_l = scores["rougeL"].fmeasure

    print("Generated:")
    print(generated)

    print("\nReference:")
    print(reference)

    print(f"\nROUGE-L: {rouge_l:.3f}")
    print("-" * 50)

    total_score += rouge_l

average_rouge_l = total_score / len(test_data)

print(f"\nAverage ROUGE-L: {average_rouge_l:.3f}")

Generated:

        Beautiful three-bedroom home located in Irvine's desirable Woodbridge community. The kitchen has been recently remodeled with quartz countertops.

Reference:

        Three-bedroom home in Irvine's Woodbridge community with a remodeled kitchen and quartz countertops.
        

ROUGE-L: 0.629
--------------------------------------------------
Generated:

        Spacious four-bedroom home in Irvine with a private swimming pool. The property features a remodeled kitchen with granite countertops.

Reference:

        Four-bedroom Irvine home featuring a private swimming pool and remodeled kitchen with granite countertops.
        

ROUGE-L: 0.686
--------------------------------------------------

Average ROUGE-L: 0.657


In [17]:
df = pd.read_csv("../data/processed/cleaned_listing_full.csv")

In [18]:
df.shape

(52794, 9)

In [19]:
df.columns.tolist()

['L_ListingID',
 'L_Address',
 'L_City',
 'beds',
 'baths',
 'price',
 'sqft',
 'remarks',
 'cleaned_remarks']

In [20]:
# Evaluation set of 20 samples
evaluation_pool= df[
    df["cleaned_remarks"].notna() &
    (df["cleaned_remarks"].str.strip() != "")
].copy()

print("Usable listings:", len(evaluation_pool))

Usable listings: 52794


In [130]:
evaluation_sample = evaluation_pool.sample(n=20, random_state=42).copy()

In [ ]:
'''evaluation_results.to_csv(
    "../data/summarization_evaluation.csv",
    index=False
)'''

In [21]:
evaluation_results = pd.read_csv("../data/summarization_evaluation.csv")

In [23]:
evaluation_results.head()

,cleaned_remarks,generated_summary,reference_summary
0,Introducing an extraordinary opportunity nestl...,"This stunning 7,100-square-foot architectural ...",This expansive estate boasts 7 bedrooms and 6 ...
1,"East-facing and beautifully upgraded, this exc...","East-facing and beautifully upgraded, this exc...","East-facing and beautifully upgraded, this exc..."
2,Placentia welcomes you to this beautifully upd...,Placentia welcomes you to this beautifully upd...,Placentia welcomes you to this beautifully upd...
3,Just 3 Blocks from the Cayucos Pier with Spect...,Just 3 Blocks from the Cayucos Pier with Spect...,Welcome to your dream beach retreat-this rare ...
4,"Welcome to this well maintained 3-bedroom, 1 b...","Welcome to this well maintained 3-bedroom, 1 b...","Welcome to this well maintained 3-bedroom, 1 b..."


In [24]:
evaluation_results.isna().sum()

cleaned_remarks      0
generated_summary    0
reference_summary    0
dtype: int64

In [26]:
scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

evaluation_results["rouge_l"] = evaluation_results.apply(
    lambda row: scorer.score(
        row["reference_summary"],
        row["generated_summary"]
    )["rougeL"].fmeasure,
    axis=1
)

In [29]:
evaluation_results['rouge_l'].describe()

count    20.000000
mean      0.696717
std       0.297376
min       0.095238
25%       0.509673
50%       0.688038
75%       1.000000
max       1.000000
Name: rouge_l, dtype: float64

In [30]:
lowest_scores = evaluation_results.sort_values(
    "rouge_l"
).head(5)

for _, row in lowest_scores.iterrows():
    print("=" * 80)
    print(f"ROUGE-L: {row['rouge_l']:.3f}")

    print("\nGenerated:")
    print(row["generated_summary"])

    print("\nReference:")
    print(row["reference_summary"])

ROUGE-L: 0.095

Generated:
Buyer couldn't perform!!! This home is a great investment and definitely counted as one of the gem stones in the City of Covina.

Reference:
From private & wet saunas, koi pond, waterfall, luxurious pool, jacuzzi to having your own basketball court, indoor raquet ball court, indoor firepit and home theater. This home has its own automated gate entrance for its own privacy and security.
ROUGE-L: 0.126

Generated:
The kitchen is just off the formal dining room and features granite counters, built-in microwave, ample cabinet space, a light and bright breakfast nook with additional breakfast bar and kitchen desk, and a door to the rear patio for easy outdoor entertaining. The two car garage is attached and has built-in cabinets plus room for a golf cart.

Reference:
Single level end unit located in The Villages Golf & Country Club, an active, gated, 55 community. Two bedrooms, each with their own bath, two and a half bathrooms, and a den provide comfort and conve

In [53]:
evaluation_results["qualitative_score"] = [
    2,  # North Tustin
    2,  # Jordan Ranch
    2,  # Placentia
    2,  # Cayucos
    2,  # ADU
    2,  # Irvine condo
    2,  # Santa Fe Springs
    2,  # Riverside
    1,  # Claremont
    2,  # Wildomar
    2,  # Palm Springs
    2,  # Villages
    2,  # Canyon Crest
    2,  # Downtown LA
    1,  # Costa Mesa
    2,  # Moreno Valley
    2,  # Oxnard
    0,  # Covina
    2,  # Monrovia
    2   # Acton
]

In [56]:
'''evaluation_results.to_csv(
    "../data/summarization_evaluation.csv",
    index=False
)'''

'evaluation_results.to_csv(\n    "../data/summarization_evaluation.csv",\n    index=False\n)'

In [57]:
evaluation_results = pd.read_csv("../data/summarization_evaluation.csv")

In [60]:
evaluation_results.isna().sum()

cleaned_remarks      0
generated_summary    0
reference_summary    0
rouge_l              0
qualitative_score    0
dtype: int64

In [61]:
evaluation_results['qualitative_score'].value_counts()

qualitative_score
2    17
1     2
0     1
Name: count, dtype: int64

In [64]:
evaluation_results['qualitative_score'].value_counts(normalize=True)

qualitative_score
2    0.85
1    0.10
0    0.05
Name: proportion, dtype: float64

In [68]:
total_score = evaluation_results["qualitative_score"].sum()
max_score = len(evaluation_results) * 2

percentage = (total_score / max_score) * 100

print(f"Total score: {total_score}/{max_score}")
print(f"Qualitative evaluation score: {percentage:.1f}%")

Total score: 36/40
Qualitative evaluation score: 90.0%


For the Answerability layer, before the agent tries to answer a query, determine whether the available real-estate data can actually answer it, and if not, explain why.

Answerable ex: "Find me 3-bedroom homes in Irvine under $1 million."
<- The system recognizes that bedrooms, city, and price are available fields.

Not answerable ex: "Which Irvine home has the best school district?"
<- The MLS data does not contain this kind of school-related information. 

In [8]:
project_root = os.path.abspath('../')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [9]:
from scripts.w4_queryparser import QueryParser, SchemaValidator

In [34]:
class AnswerabilityChecker: 
    def __init__(self, taxonomy_path): 
        with open(taxonomy_path, "r", encoding="utf-8") as f:
            self.taxonomy = json.load(f)

        self.validator = SchemaValidator()

        self.real_estate_keywords = ['house', 'home', 'bed', 'bath', 
        'property', 'listing', 'price', 'sqft', 'pool', 'garage'] 

    def check_pre_query(self, query): 
        """Check BEFORE generating SQL""" 
        query_lower = query.lower() 

        # Check 1: Is this a real estate question? 
        has_re_terms = any(
            kw in query_lower 
            for kw in self.real_estate_keywords
        ) 

        if not has_re_terms: 
            return False, "This doesn't appear to be a real estate question" 

        # Check 2: Does query reference valid data? 
        # (Use Week 4's schema validator) 
        parser = QueryParser()

        try:
            filters = parser.parse(query)
        except (ValueError, KeyError, TypeError):
            return False, "I couldn't identify a supported property criterion in your question."

        # Check 3: Make sure something was actually extracted
        if not filters:
            return False, (
                "I couldn't identify a supported property criterion "
                "in your question."
            )

        # Check 4: Validate parsed filters
        try:
            self.validator.validate_query(filters)
        except ValueError as e:
            return False, f"Query references invalid data: {e}" 

        return True, "Query is answerable" 

    def check_post_query(self, results_df): 
        """Check AFTER executing SQL""" 
        if len(results_df) == 0: 
            return False, "No listings match your criteria" 

        # Check for all-null results 
        if results_df.isnull().all().all(): 
            return False, "Query returned no meaningful data" 

        return True, "Results found" 

In [35]:
# Usage: 
checker = AnswerabilityChecker(taxonomy_path=TAXONOMY_PATH) 

In [22]:
# Test case
user_query = "Find me a 3 bedroom home in Irvine"

can_answer, message = checker.check_pre_query(user_query) 

In [23]:
print("Answerable:", can_answer)
print("Message:", message)

Answerable: True
Message: Query is answerable


In [36]:
def process_query(user_query):

    # 1. Check whether query is answerable
    can_answer, message = checker.check_pre_query(user_query)

    if not can_answer:
        return {
            "error": message,
            "answerable": False
        }

    # 2. Parse query
    parser = QueryParser()
    filters = parser.parse(user_query)

    # 3. Convert filters to SQL
    sql, params = parser.to_sql(filters)

    # 4. Execute SQL
    results = execute_query(sql, params)

    # 5. Check results
    can_answer, message = checker.check_post_query(results)

    if not can_answer:
        return {
            "message": message,
            "answerable": False,
            "results": []
        }

    # 6. Return results
    return {
        "answerable": True,
        "message": message,
        "results": results
    }


def execute_query(sql, params):
    conn = get_connection()
    cursor = conn.cursor(dictionary=True)

    cursor.execute(sql, params)
    results = cursor.fetchall()

    cursor.close()
    conn.close()

    return pd.DataFrame(results)



In [18]:
# Test 1: Valid real-estate query
user_query = "Find me 3 bedroom homes in Irvine"

result = process_query(user_query)

print(result)

{'answerable': True, 'message': 'Results found', 'results':          id L_ListingID L_DisplayId         L_Address  L_Zip  \
0       600  1118294602  1118294602        320 Bronze  92618   
1     46604  1119964557  1119964557          46 Eagle  92604   
2     57019  1139811520  1139811520   5200 Irvine 254  92620   
3     61812  1142070303  1142070303   142 Hedge Bloom  92618   
4     67145  1144648212  1144648212   81 Island Coral  92620   
..      ...         ...         ...               ...    ...   
216  271105  1174648555  1174648555  13 Sacramento 95  92604   
217  271244  1174668774  1174668774    13 Firestone 5  92614   
218  271621  1157633849  1157633849            5 Eden  92620   
219  271674  1154236101  1154236101          3 Varesa  92620   
220  271835  1174654486  1174654486     59 Bower Tree  92603   

                 LM_char10_70 L_AddressStreet  L_City L_State      L_Class  \
0          Santa Cruz (SCRUZ)          Bronze  Irvine      CA  Residential   
1              

In [19]:
len(result["results"])

221

In [20]:
user_query = "What is the capital of France?"

result = process_query(user_query)

print(result)

{'error': "This doesn't appear to be a real estate question", 'answerable': False}


In [ ]:
# Identified as a real estate question, but the MLS data does not contain the information asked for
user_query = "Find homes with a unicorn room"

result = process_query(user_query)

print(result)

{'error': "I couldn't identify a supported property criterion in your question.", 'answerable': False}


In [24]:
user_query = "What is the weather in Irvine?"

result = process_query(user_query)

print(result)

{'error': "This doesn't appear to be a real estate question", 'answerable': False}


In [25]:
user_query = "Find homes in Atlantis"

result = process_query(user_query)

print(result)

{'error': "I couldn't identify a supported property criterion in your question.", 'answerable': False}


In [ ]:
# There are no listings with 20 bedrooms
user_query = "Find homes with 20 bedrooms"

result = process_query(user_query)

print(result)

{'message': 'No listings match your criteria', 'answerable': False, 'results': []}


In [38]:
user_query = "Find homes with a pool"

result = process_query(user_query)

print(result)

{'answerable': True, 'message': 'Results found', 'results':           id L_ListingID L_DisplayId                     L_Address  L_Zip  \
0        293  1119909544  1119909544              38058 Highway 94  91905   
1        406  1118349692  1118349692          47645 Calle Diamante  92201   
2        411  1118348894  1118348894       79140 Fred Waring Drive  92203   
3        608  1136866728  1136866728             520 Foothill Road  93023   
4        611  1128160382  1128160382                3 Island Vista  92657   
...      ...         ...         ...                           ...    ...   
7319  272242  1174729710  1174729710  15547 Valley Vista Boulevard  91436   
7320  272249  1174731430  1174731430             115 Shirley Court  92324   
7321  272250  1174731304  1174731304                   750 Jericho  92028   
7322  272253  1174731103  1174731103            69 Dartmouth Drive  92270   
7323  272264  1174732751  1174732751            1033 S Calle Rolph  92264   

              L